# SNN Evaluation on Real-Time Data
In this notebook, we test the performance of trained SNNs on a real-time stream of data.
Based (or copied) from Mesquita's [repository](https://github.com/monkin77/snn-torch/blob/master/src/hfo/5_detection/hfo_evaluation.ipynb)

In [1]:

# Show current directory
import os
curr_dir = os.getcwd()
print(curr_dir)

from collections import deque

c:\Users\NCN\Documents\PedroFelix\LAVA_SNN_ripples\snnTorch


## Add Parent Directory to Path

In [2]:
import sys
parent_dir = os.path.abspath(os.path.join(curr_dir, os.pardir))
# Add the grandparent directory to the system path
# grandparent_dir = os.path.abspath(os.path.join(curr_dir, os.pardir, os.pardir))
sys.path.append(parent_dir)
print(sys.path)

['c:\\nrn\\lib\\python', 'c:\\Users\\NCN\\Miniconda3\\envs\\lava_snn_ripples\\python39.zip', 'c:\\Users\\NCN\\Miniconda3\\envs\\lava_snn_ripples\\DLLs', 'c:\\Users\\NCN\\Miniconda3\\envs\\lava_snn_ripples\\lib', 'c:\\Users\\NCN\\Miniconda3\\envs\\lava_snn_ripples', '', 'c:\\Users\\NCN\\Miniconda3\\envs\\lava_snn_ripples\\lib\\site-packages', 'c:\\Users\\NCN\\Miniconda3\\envs\\lava_snn_ripples\\lib\\site-packages\\win32', 'c:\\Users\\NCN\\Miniconda3\\envs\\lava_snn_ripples\\lib\\site-packages\\win32\\lib', 'c:\\Users\\NCN\\Miniconda3\\envs\\lava_snn_ripples\\lib\\site-packages\\Pythonwin', 'c:\\Users\\NCN\\Documents\\PedroFelix\\LAVA_SNN_ripples']



## Check if Cuda is available


In [3]:
import torch
import numpy as np

# Check CUDA Installation
print(torch.cuda.is_available())

# Get the number of available GPUs
num_gpus = torch.cuda.device_count()
print(f"Number of GPUs: {num_gpus}")

# Get information about each GPU
for i in range(num_gpus):
    device_props = torch.cuda.get_device_properties(i)
    print(f"\nGPU {i}:")
    print(f"  Name: {device_props.name}")
    print(f"  Total memory: {device_props.total_memory / 1024**3:.2f} GB")
    print(f"  Multiprocessor count: {device_props.multi_processor_count}")
    print(f"  Major compute capability: {device_props.major}")
    print(f"  Minor compute capability: {device_props.minor}")

False
Number of GPUs: 0



## Define the Device that will be used to train the SNN


In [4]:


# Set the device to be used
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")     # torch.device("cpu") #

print("device: ", device)



device:  cpu



## Define Problem and Simulation Parameters


In [5]:
# ----- Simulation Time Parameters -----
dt = 1                         # Time between two timesteps (ms), a.k.a. virtual time step interval. (NOT ALIGNED WITH THE SAMPLING RATE OF THE INPUT DATA (2048 Hz)
init_offset = 0 # 900 # 33400      #   

In [6]:
# unit: timesteps (ms) - The time window after the GT annotation where the network should predict the burst (GT_time, GT_time + PRED_CAUSALITY_WINDOW)
# This is needed to give the network some extra time steps to increase the membrane potential and spike
RIPPLE_DETECTION_OFFSET = [18, 45, 31, 20]
PRED_CAUSALITY_WINDOW = int(5)     # Giving PRED_CAUSALITY_WINDOW ms for the network to update its inner state and spike  
# in timesteps (ms) - Max time from the Insertion Timing to the GT annotation

add_tolerance=False

TOLERANCE= RIPPLE_DETECTION_OFFSET[3] if add_tolerance else 0 # in timesteps (ms) - Tolerance for the prediction window, to account for the fact that the network might not be able to predict the burst exactly at the GT annotation time

MAX_DETECTION_OFFSET = int(RIPPLE_DETECTION_OFFSET[2] + RIPPLE_DETECTION_OFFSET[1]+ PRED_CAUSALITY_WINDOW + TOLERANCE)  # in timesteps (ms) - Max time from the Insertion Timing to the GT annotation

print(f"PRED_CAUSALITY_WINDOW: {PRED_CAUSALITY_WINDOW}")
print(f"MAX_DETECTION_OFFSET: {MAX_DETECTION_OFFSET} ms")

PRED_CAUSALITY_WINDOW: 5
MAX_DETECTION_OFFSET: 81 ms



## Read the Input Data and the Ground Truth
Here the data will be just the spikified 8 channel-data for the last 20% of each dataset.

As such, it shall be read farther ahead, during the testing process...



In [7]:
#### CHOOSE A SEED FOR REPRODUCIBILITY
SEEDED = False # Set to True if you want to use a specific seed for reproducibility
if SEEDED:
    seed = 0
    time_duration= 60 # in seconds, the duration of the test
    window=np.arange(seed*time_duration*1000, (time_duration+seed*time_duration)*1000, 1) # 1000 ms window

## Create the Dataset and Dataloader to user tensor-ready data


In [8]:
# identifier="30000_1000_100"
identifier="1000_200_median"
dataset_path=os.path.join(parent_dir,"extract_Nripples","train_pedro","dataset_up_down",identifier)




## Define the SNN Architecture
Similar to what we trained before

In [9]:
import snntorch as snn
import torch.nn as nn
from snntorch import surrogate

# Global Parameters
v_thr = 1.0
placeholder_val = 0.5

# Define the surrogate gradient function to propagate spikes through the network
spike_grad = surrogate.fast_sigmoid()   # surrogate.atan()   

In [10]:


# Parameters for Dense Layers
inputDataDim = 2       # max_channel_idx - min_channel_idx + 1    # Number of input channels

input_to_hidden = (inputDataDim, 24) # 16 # TODO: Increase the size of this layer # (inputDataDim, 100) # (inputDataDim, 500)  # Number of neurons in the first Fully-Connected Layer

hiddenL2Dim = (input_to_hidden[1], input_to_hidden[1])  # Number of neurons in the Recurrent Fully-Connected Layer (L2)

hiddenL3Dim = (input_to_hidden[1], 16)  # Number of neurons in the Fully-Connected Layer (L3)

hiddenL4Dim = (hiddenL3Dim[1], input_to_hidden[1])  # Number of neurons in the Recurrent Fully-Connected Layer (L4)

hidden_to_out = (hiddenL3Dim[1], 1)  # Number of neurons in the Output Fully-Connected Layer
# In this case, we only need 1 output neuron -> Fires when HFO is detected



In [11]:
# Define Network
class Net(nn.Module):
    def __init__(self):
        super().__init__()

        # Initialize layers
        
        # Create a Linear Layer to serve input to LIF1
        self.fc_in = nn.Linear(input_to_hidden[0], input_to_hidden[1],
                bias=False,
                dtype=torch.float32     # Set the data type of the weights to float32
        )

        # TODO: Should the LIF neurons be able to get a negative membrane potential? I think so?
        self.lif1 = snn.Synaptic(
            alpha=torch.full(size=(input_to_hidden[1],), fill_value=placeholder_val), 
            beta=torch.full(size=(input_to_hidden[1],), fill_value=placeholder_val),
            threshold=v_thr,
            reset_mechanism="zero", reset_delay=False,
            # TODO: How to add Refractory Period?
            # init_hidden=True,   # enables the methods in snntorch.backprop to automatically clear the hidden states and detach them from the comp. graph
            spike_grad=spike_grad,
            learn_alpha=True,   # Learn the alpha parameter
            learn_beta=True,    # Learn the beta parameter
            learn_threshold=False,   # Learn the threshold parameter
            
        )      

        """ self.fc2 = nn.Linear(
            hiddenL2Dim[0], hiddenL2Dim[1],
            bias=False,
            dtype=torch.float32     # Set the data type of the weights to float32
        ) """

        self.fc3 = nn.Linear(
            hiddenL3Dim[0], hiddenL3Dim[1],
            bias=False,
            dtype=torch.float32     # Set the data type of the weights to float32
        )

        self.lif2 = snn.Synaptic(
            alpha=torch.full(size=(hiddenL3Dim[1],), fill_value=placeholder_val), 
            beta=torch.full(size=(hiddenL3Dim[1],), fill_value=placeholder_val),
            threshold=v_thr,
            reset_mechanism="zero", reset_delay=False,
            # TODO: How to add Refractory Period?
            # init_hidden=True,   # enables the methods in snntorch.backprop to automatically clear the hidden states and detach them from the comp. graph
            spike_grad=spike_grad,
            learn_alpha=True,   # Learn the alpha parameter
            learn_beta=True,    # Learn the beta parameter
            learn_threshold=False,   # Learn the threshold parameter
        )   

        """ self.fc4 = nn.Linear(
            hiddenL4Dim[0], hiddenL4Dim[1],
            bias=False,
            dtype=torch.float32     # Set the data type of the weights to float32
        ) """

        self.fc_out = nn.Linear(
            hidden_to_out[0], hidden_to_out[1],
            bias=False,
            dtype=torch.float32     # Set the data type of the weights to float32
        )

        self.lif_out = snn.Synaptic(
            alpha=placeholder_val, 
            beta=placeholder_val,
            threshold=v_thr,
            reset_mechanism="zero", reset_delay=False,
            # init_hidden=True,   # enables the methods in snntorch.backprop to automatically clear the hidden states and detach them from the comp. graph
            spike_grad=spike_grad,
            learn_alpha=True,   # Learn the alpha parameter
            learn_beta=True,    # Learn the beta parameter
            learn_threshold=False,   # Learn the threshold parameter
        )

        # Initialize the membrane potential of each LIF neuron
        self.syn1, self.mem1, self.spk1 = None, None, None
        self.syn2, self.mem2, self.spk2 = None, None, None
        self.syn_out, self.mem_out, self.spk_out = None, None, None

    """
    Function called during the forward pass of the network
    """
    def forward(self, x: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        '''
        Forward Pass of the Network (Single Step Update)

        Parameters:
        - x: input tensor. Shape: (batch_size, num_features)

        Returns:
        - spk_vals: tuple of tensors containing the spikes of the neurons. Shape: (batch_size, num_neurons)
        - mem_vals: tuple of tensors containing the membrane potentials of the neurons. Shape: (batch_size, num_neurons)
        - syn_vals: tuple of tensors containing the currents of the neurons. Shape: (batch_size, num_neurons)
        '''
        cur_batch_size, cur_num_channels = x.shape

        # --- Lazy State Initialization
        if self.mem1 is None:
            device = x.device   # Get the device of the input tensor

            # Initialize the membrane potential of each LIF neuron
            self.syn1, self.mem1 = self.lif1.reset_mem()
            self.syn2, self.mem2 = self.lif2.reset_mem()
            self.syn_out, self.mem_out = self.lif_out.reset_mem()

            # # Define small residual for spk1
            # spk1_factor = 0.01
            # self.spk1 = torch.rand(size=(cur_batch_size, input_to_hidden[1]), dtype=torch.float32, device=device) * spk1_factor
            # self.spk2 = torch.zeros(size=(cur_batch_size, hiddenL3Dim[1]), dtype=torch.float32, device=device) * spk1_factor
            # self.spk_out = torch.zeros(size=(cur_batch_size, hidden_to_out[1]), dtype=torch.float32, device=device)

        # 
        if len(x.shape) == 1:
            # If the input is 1D, it means we have only one feature (one channel)
            # Unsqueeze the input to add the num_features dimension
            x = x.unsqueeze(1)
            
        ############# State Update #############
        # Calculate Input Current for LIF1 from the Input Layer (FC1) Input -> LIF1
        cur_fc1 = self.fc_in(x) 
    
        # Calculate Input Current from Recurrent Layer (FC2) LIF1 -> LIF1
        # cur_fc2 = self.fc2(spk1)   # Connect LIF1 to itself using FC Layer 2 (Recurrent Layer)

        # Join the input currents for LIF1 (FC1 + FC2)
        cur1 = cur_fc1 # + cur_fc2  # TODO: Not feeding Recurent Layer to LIF1 for now

        # Feed the joined input current to LIF1
        self.spk1, self.syn1, self.mem1 = self.lif1(cur1, self.syn1, self.mem1)  # Feed input to LIF1

        # Calculate Input Current for LIF2 from LIF1 (FC3) LIF1 -> LIF2
        cur2 = self.fc3(self.spk1)   # Connect LIF1 to LIF2 using FC Layer 3
        # Feed the input current to LIF2 and get the spikes, synaptic currents and membrane potentials
        self.spk2, self.syn2, self.mem2 = self.lif2(cur2, self.syn2, self.mem2)  # Feed input to LIF2

        # Calculate Input Current for LIF_OUT from LIF2 (FC4) LIF2 -> LIF_OUT
        cur_out = self.fc_out(self.spk2)
        # Feed the input current to LIF_OUT and get the spikes, synaptic currents and membrane potentials
        self.spk_out, self.syn_out, self.mem_out = self.lif_out(cur_out, self.syn_out, self.mem_out)  # Feed input to LIF_OUT

        # Return the currents, membrane potentials and spikes of the current timestep
        syn_val = (self.syn1, self.syn2, self.syn_out)
        mem_vals = (self.mem1, self.mem2, self.mem_out)
        spk_vals = (self.spk1, self.spk2, self.spk_out)

        # TODO: Check if the dimensions are correct
        return (spk_vals, mem_vals, syn_val)


## Load the Trained Network


In [12]:
# Load the network onto CUDA if available
net = Net().to(device)

prefix="dsb4updn_median_200_2b"
# prefix="updnb4ds_100_10"
# Load the Trained Parameters from a file
net_filename = f"out/{prefix}_trained_net_loss.pth"  # trained_net_loss_penalty.pth
net.load_state_dict(torch.load(net_filename, map_location=device))

# Set the network to evaluation mode
net.eval()



Net(
  (fc_in): Linear(in_features=2, out_features=24, bias=False)
  (lif1): Synaptic()
  (fc3): Linear(in_features=24, out_features=16, bias=False)
  (lif2): Synaptic()
  (fc_out): Linear(in_features=16, out_features=1, bias=False)
  (lif_out): Synaptic()
)

## Show the Network Parameters

In [13]:


# Display the network architecture
total_params = 0    # Accumulator for the total params in the network
# Iterate through the layers of the network
for idx, (name, param) in enumerate(net.named_parameters()):
    # print("param: ", param)
    if param.shape == torch.Size([]):
        print(f"Scalar Param ({name}) | Shape={param.shape} | Value={param} \n")
    elif len(param.shape) == 1:
        print(f"Vector Param ({name}) | Shape={param.shape} | Value={param} \n")
    else:
        print(f"Tensor Param ({name}) | Shape={param.shape}. Total={param.numel()} Preview: {param[:8, :8]}\n")

    # Add the number of parameters in the layer to the total
    total_params += param.numel()

# Print the total number of parameters in the network
print(f"Total Parameters: {total_params}")


Tensor Param (fc_in.weight) | Shape=torch.Size([24, 2]). Total=48 Preview: tensor([[-0.2231, -0.3006],
        [ 0.0195, -0.4128],
        [ 0.6878, -0.0312],
        [ 0.5308,  0.2855],
        [ 0.4794, -0.0555],
        [-0.1724, -0.0942],
        [-0.4212, -0.0222],
        [ 0.7376, -0.2642]], grad_fn=<SliceBackward0>)

Vector Param (lif1.beta) | Shape=torch.Size([24]) | Value=Parameter containing:
tensor([0.2816, 0.5795, 0.5708, 0.1441, 0.3856, 0.7997, 0.6325, 0.5325, 0.6160,
        0.8294, 0.4760, 0.4622, 0.2794, 0.4109, 0.7209, 0.7471, 0.2293, 0.9739,
        0.4441, 0.9446, 0.3754, 0.3657, 0.5112, 0.5534], requires_grad=True) 

Vector Param (lif1.alpha) | Shape=torch.Size([24]) | Value=Parameter containing:
tensor([0.5283, 0.4616, 0.6705, 0.4787, 0.6331, 0.4849, 0.4551, 0.6690, 0.5003,
        0.5609, 0.4566, 0.8171, 0.3948, 0.4447, 0.5143, 0.5062, 0.4491, 0.7028,
        0.7726, 0.6544, 0.8609, 0.3055, 0.1984, 0.6553], requires_grad=True) 

Tensor Param (fc3.weight) | Shape=

## Feed the Data in Real-Time

In [14]:


"""
Tensor to store the next timestep when each output neuron may spike (after the refractory period).
When a neuron spikes, it will be set to the current timestep + refractory period.
"""

refrac_period = 200  # Refractory period in timesteps (ms) for the output layer
lif_out_refrac_times = torch.full(size=(hidden_to_out[1],), fill_value=0.0, device=device)

print("lif_out_refrac_times: ", lif_out_refrac_times)



lif_out_refrac_times:  tensor([0.])


In [15]:

'''
# Active GT Time-To-Live (TTL). This is a counter that decrements every timestep until it reaches 0.
If it reaches 0, it means the Network failed to predict the HFO within the GT tolerance window.
None -> No GT Event is active
'''

from torch.utils.data import TensorDataset, DataLoader
all_metrics = {}

# gt_tensor = torch.from_numpy(ripples_start).to(device)
# gt_tensor = torch.from_numpy(ripples_start).to(device)

f1_over_time=[ ] # List to store the F1 score over time   
out_spikes=[]

# Disable gradient calculation for inference
for dataset in os.listdir(dataset_path):
    data_path= os.path.join(dataset_path, dataset)
    print(f"Processing dataset: {data_path}")
    data=np.load(os.path.join(data_path, "spike_data.npy"))  # Load the input data (UP/DN spikes)
    ripples=np.load(os.path.join(data_path, "ripples.npy"))  # Load the GT data (HFO Insertion Timing)
    all_metrics[dataset] = {}  # Initialize channel metrics for this dataset
    ripples_start = ripples[:, 0]  # Get the GT Insertion Timing (first column of the ripples array)
    if SEEDED:
        # If the data is seeded, we need to slice the data to the desired time window
        data = data[window,:]
        ripples_window = []
        for ripple in ripples_start:
            if ripple >= window[0] and ripple <= window[-1]:
                    ripples_window.append(ripple)
        ripples_start=np.array(ripples_window)-window[0]-TOLERANCE  # Adjust the GT Insertion Timing to the new window
    else:
        ripples_start=ripples_start-TOLERANCE
    num_hfo_events = len(ripples_start)
    input_tensor = torch.tensor(data,dtype=torch.float32).to(device)
    gt_tensor=torch.tensor(ripples_start,dtype=torch.float32).to(device)

    
    for channel in range(data.shape[1]):
        # Normalize the data for each channel
        # Convert numpy arrays to PyTorch tensors and move them to the selected device
        curr_gt_idx = 0 # Index of the current GT event
        curr_gt = None  # Stores the current GT event (GT Insertion Timing)
        active_gts = deque()  # Queue of active GT events: each item is (gt_time, remaining_ttl)
        output_spikes_channel=[]
        TP,FP,FN,TN=0,0,0,0
        total_num_steps= input_tensor.shape[0]  # Total number of timesteps in the input data
        print(f"Processing channel {channel} with {num_hfo_events} HFO events")
        lif_out_refrac_times[0]=-10000
        with torch.no_grad():
            for step in range(total_num_steps):
                # Get the current input (UP/DN spikes)
                curr_input = input_tensor[step,channel,:]

                # Unsqueeze the input to add the num_batches dimension
                curr_input = curr_input.unsqueeze(0)
                # print(f"curr_input: {curr_input}")

                ADDED_FN = False    # Tracks if a False Negative was added in this timestep

                # # Check if a GT event leaves the detection window
                # if active_gt_ttl is not None and active_gt_ttl < 0:
                #     # GT Event Expired
                #     print(f"GT Event expired at timestep {step} with GT Insertion Timing: {curr_gt}")

                #     # Add a False Negative to the Confusion Matrix
                #     FN += 1
                #     # Set the TTL to None
                #     active_gt_ttl = None
                #     # Move to the next GT event
                #     curr_gt_idx += 1
                #     # Set ADDED_FN to True
                #     ADDED_FN = True

                # Check if a GT event enters the detection window
                if curr_gt_idx < num_hfo_events :
                    curr_gt = int(gt_tensor[curr_gt_idx].item())  # Get the current GT Insertion Timing
                    if curr_gt == step:
                        # Check if a GT event was already active
                        print(f"GT Event {curr_gt_idx} Insertion Timing: {curr_gt} at timestep {step}")
                        active_gts.append((curr_gt, MAX_DETECTION_OFFSET))
                        # if active_gt_ttl is not None:
                        #     raise ValueError("[Error] Two GT events detected inside the Detection Window!")

                        # GT Event starts at this timestep
                        # print(f"GT Event starts at timestep {step} with GT Insertion Timing: {curr_gt}")
                        
                        # Set the TTL to the Maximum Detection Offset from the GT Insertion Timing
                        # active_gt_ttl = MAX_DETECTION_OFFSET
                        curr_gt_idx+=1
                    
                # --------   State Update   --------
                spk, mem, syn = net(curr_input)

                # Get the spikes, membrane potentials and synaptic currents of the current timestep
                spk1, spk2, spk_out = spk
                mem1, mem2, mem_out = mem
                syn1, syn2, syn_out = syn
                # print(f"spk1: {spk1.shape} | spk2: {spk2.shape} | spk_out: {spk_out.shape}")
                # print(f"mem1: {mem1.shape} | mem2: {mem2.shape} | mem_out: {mem_out.shape}")
                # print(f"syn1: {syn1.shape} | syn2: {syn2.shape} | syn_out: {syn_out.shape}")

                if ADDED_FN:
                    """
                    If a FN was added -> GT event was not detected -> The problem formulation does not allow
                    2 HFO events to be closer together than the confidence window, so we can skip this step
                    """
                    continue

                """ if torch.sum(spk2) > 0:
                    # TODO: Remove this print statement
                    print(f"LIF2 spiked at timestep {step} with spikes: {spk2}") """
                
                # Track which GTs remain active
                new_active_gts = deque()
                for gt_time, ttl in active_gts:
                    if ttl < 0:
                        FN += 1
                        print(f"[FN] GT event at {gt_time} expired at timestep {step}")
                    else:
                        new_active_gts.append((gt_time, ttl - 1))
                active_gts = new_active_gts

                if torch.sum(spk_out) > 0:
                    # Convert spk_out to int for bitwise operations (Squeeze the batch dimension)
                    spk_out_int = spk_out.squeeze(0).int()
                    output_spikes_channel.append(step*dt)
                    # Consider the refractory period of the output neurons
                    REFRAC_STATE_MASK = lif_out_refrac_times > step * dt    # Check if each output neuron is in the refractory period
                    # Bitwise AND between the spikes by the refractory state mask
                    # Gets a mask of the neurons that spiked and are not in the refractory state
                    valid_spk_out = torch.Tensor.bool(spk_out_int & (~REFRAC_STATE_MASK))

                    # Check if any Valid Output Neuron spiked
                    if torch.sum(valid_spk_out) > 0:
                        # print(f"valid_spk_out: {valid_spk_out} | lif_out_refrac_times: {lif_out_refrac_times}")
                        # Set the refractory period for the spiking output neurons
                        lif_out_refrac_times[valid_spk_out] = float(step * dt + refrac_period)

                        # If an Output Neuron spiked -> Predicted an HFO
                        # Let's check if the predicted HFO is within the GT tolerance window
                        if active_gts:
                            # The GT event is active -> Valid Prediction
                            gt_time, _ = active_gts.popleft()
                            TP += 1
                            print(f"[TP] Spike matched GT event at timestep {step} (GT: {gt_time})")
                        else:
                            # The GT event is not active -> Invalid Prediction
                            FP += 1
                            print(f"[FP] Network detected HFO at timestep {step} without an active GT event")
                    else:
                        # No valid output neuron spiked -> No HFO Detected
                        TN += 1     # Increment True Negatives (No HFO detected)
                    
                else:
                    # If the Output Neuron did not spike -> No HFO detected
                    TN += 1     # Increment True Negatives (No HFO detected)
                if step % 100000 == 0:
                    print(f"Processed step {step}/{total_num_steps} ({(step /total_num_steps) * 100:.2f}%)")
        
        # Calculate F1
        precision = TP / (TP + FP) if (TP + FP) > 0 else 0
        recall = TP / (TP + FN) if (TP + FN) > 0 else 0
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

        # Save channel metrics under this dataset
        all_metrics[dataset][f"channel_{channel}"] = {
            "TP": int(TP),
            "FP": int(FP),
            "FN": int(FN),
            "TN": int(TN),
            "Precision": precision,
            "Recall": recall,
            "F1": f1
        }
        out_spikes.append(output_spikes_channel)

Processing dataset: c:\Users\NCN\Documents\PedroFelix\LAVA_SNN_ripples\extract_Nripples\train_pedro\dataset_up_down\1000_200_median\Amigo2_2019-07-11_11-57-07
Processing channel 0 with 224 HFO events
Processed step 0/479771 (0.00%)
[FP] Network detected HFO at timestep 412 without an active GT event
GT Event 0 Insertion Timing: 568 at timestep 568
[FN] GT event at 568 expired at timestep 650
[FP] Network detected HFO at timestep 658 without an active GT event
[FP] Network detected HFO at timestep 858 without an active GT event
GT Event 1 Insertion Timing: 969 at timestep 969
[FN] GT event at 969 expired at timestep 1051
[FP] Network detected HFO at timestep 1304 without an active GT event
GT Event 2 Insertion Timing: 1629 at timestep 1629
[TP] Spike matched GT event at timestep 1637 (GT: 1629)
[FP] Network detected HFO at timestep 1880 without an active GT event
[FP] Network detected HFO at timestep 2707 without an active GT event
[FP] Network detected HFO at timestep 3552 without an a

In [16]:
for dataset in all_metrics.keys():
    if dataset=="overall" or dataset=="metrics":
        continue
    print(dataset)
    dataset_TP = 0
    dataset_FP = 0
    dataset_FN = 0
    dataset_TN = 0

    for ch, metrics in all_metrics[dataset].items():
        if ch == "summary":
            continue  # Skip the summary itself
        if isinstance(metrics, dict) and {"TP", "FP", "FN", "TN"}.issubset(metrics):
            dataset_TP += metrics["TP"]
            dataset_FP += metrics["FP"]
            dataset_FN += metrics["FN"]
            dataset_TN += metrics["TN"]

    # Compute metrics
    precision = dataset_TP / (dataset_TP + dataset_FP) if (dataset_TP + dataset_FP) > 0 else 0
    recall = dataset_TP / (dataset_TP + dataset_FN) if (dataset_TP + dataset_FN) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

    # Store in summary
    all_metrics[dataset]["summary"] = {
        "TP": dataset_TP,
        "FP": dataset_FP,
        "FN": dataset_FN,
        "TN": dataset_TN,
        "Precision": precision,
        "Recall": recall,
        "F1": f1
    }
    print(all_metrics[dataset]["summary"])

Amigo2_2019-07-11_11-57-07
{'TP': 949, 'FP': 3537, 'FN': 843, 'TN': 3833682, 'Precision': 0.21154703522068657, 'Recall': 0.5295758928571429, 'F1': 0.3023255813953488}
Dlx1_2021-02-12_12-46-54
{'TP': 155, 'FP': 2016, 'FN': 93, 'TN': 1631965, 'Precision': 0.0713956701980654, 'Recall': 0.625, 'F1': 0.12815212897891692}
Som2_2019-07-24_12-01-49
{'TP': 269, 'FP': 1902, 'FN': 243, 'TN': 1655829, 'Precision': 0.1239060340856748, 'Recall': 0.525390625, 'F1': 0.20052180395080133}
Thy7_2020-11-11_16-05-00
{'TP': 574, 'FP': 659, 'FN': 538, 'TN': 1189495, 'Precision': 0.46553122465531227, 'Recall': 0.5161870503597122, 'F1': 0.4895522388059702}


In [17]:
# Global metrics across all datasets
global_TP = sum(all_metrics[d]["summary"]["TP"] for d in all_metrics if d != "overall" and d !=  "metrics" and d != "parameters")
global_FP = sum(all_metrics[d]["summary"]["FP"] for d in all_metrics if d != "overall" and d !=  "metrics" and d != "parameters")
global_FN = sum(all_metrics[d]["summary"]["FN"] for d in all_metrics if d != "overall" and d !=  "metrics" and d != "parameters")
global_TN = sum(all_metrics[d]["summary"]["TN"] for d in all_metrics if d != "overall" and d !=  "metrics" and d != "parameters")

global_precision = global_TP / (global_TP + global_FP) if (global_TP + global_FP) > 0 else 0
global_recall = global_TP / (global_TP + global_FN) if (global_TP + global_FN) > 0 else 0
global_f1 = 2 * global_precision * global_recall / (global_precision + global_recall) if (global_precision + global_recall) > 0 else 0

# Store in top-level dictionary
all_metrics["overall"] = {
    "TP": global_TP,
    "FP": global_FP,
    "FN": global_FN,
    "TN": global_TN,
    "Precision": global_precision,
    "Recall": global_recall,
    "F1": global_f1
}
print(all_metrics["overall"])

{'TP': 1947, 'FP': 8114, 'FN': 1717, 'TN': 8310971, 'Precision': 0.19351953086174337, 'Recall': 0.5313864628820961, 'F1': 0.2837158469945355}


In [18]:
total_predictions = all_metrics["overall"]["TP"] + all_metrics["overall"]["FP"] + all_metrics["overall"]["FN"] + all_metrics["overall"]["TN"]
print(f"Total Predictions: {total_predictions}")
print

Total Predictions: 8322749


<function print>

In [19]:

import json

# Export the results to a JSON file
OUTPUT_FOLDER = f"eval/"
# create the output folder if it doesn't exist
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

# Create a dictionary with the results
all_metrics["parameters"] = {
    "ripple_detection_offset": RIPPLE_DETECTION_OFFSET,
    "pred_causality_window": PRED_CAUSALITY_WINDOW,
    "max_detection_offset": MAX_DETECTION_OFFSET,
    "refractory_period_gt": refrac_period,
    "tolerance": TOLERANCE,
    "seed": seed if SEEDED else None,
    "time_duration": time_duration if SEEDED else None,
    }

EXPORT_JSON_FILE = True
if EXPORT_JSON_FILE:
    if SEEDED:  
        metrics_file_name = f"{OUTPUT_FOLDER}/{prefix}_results_seed{seed}.json"
    else:
        metrics_file_name = f"{OUTPUT_FOLDER}/{prefix}_results.json"
    with open(metrics_file_name, 'w') as f:
        json.dump(all_metrics, f, indent=4)

In [20]:
# Find the longest spike sequence
max_len = max(len(ch_spikes) for ch_spikes in out_spikes)

# Pad each row with zeros to the same length
out_spikes_padded = np.array([
    np.pad(ch_spikes, (0, max_len - len(ch_spikes)), mode='constant')
    for ch_spikes in out_spikes
])



print(f"Output Spikes Shape: {out_spikes_padded.shape}")
EXPORT_SPIKE_FILE = True
SPIKE_FOLDER= f"eval/spikes/"
if EXPORT_SPIKE_FILE:
    if SEEDED:
        spike_file_name = f"{SPIKE_FOLDER}/{prefix}_spikes_seed{seed}.npy"
    else:
        spike_file_name = f"{SPIKE_FOLDER}/{prefix}_spikes.npy"
    # create the output folder if it doesn't exist
    os.makedirs(SPIKE_FOLDER, exist_ok=True)
    np.save(spike_file_name, out_spikes_padded)

Output Spikes Shape: (32, 11279)


In [21]:
# import matplotlib.pyplot as plt
# datasets=os.listdir(dataset_path)
# # get the first dataset
# dataset = datasets[0]
# # Load the GT data for the first dataset
# ripples = np.load(os.path.join(dataset_path, dataset, "ripples.npy"))  # Load the GT data (HFO Insertion Timing)
# ripples_start = ripples[:, 0]  # Get the GT Insertion Timing (first column of the ripples array)
# # Load the input data for the first dataset
# data = np.load(os.path.join(dataset_path, dataset, "spike_data.npy"))  # Load the input data (UP/DN spikes)
# filtered=np.load(os.path.join(dataset_path, dataset, "filtered_data.npy"))  # Load the filtered data (if available)
# channel=1
# window=np.arange(500, 1000, 1)  # Default window is the entire data length
# if SEEDED:
#     # If the data is seeded, we need to slice the data to the desired time window
#     data = data[window,:,:]
#     filtered = filtered[window,:]
#     ripples_window = []
#     for ripple in ripples_start:
#         if ripple >= window[0] and ripple <= window[-1]:
#                 ripples_window.append(ripple)
#     ripples_start=np.array(ripples_window)-window[0]-TOLERANCE  # Adjust the GT Insertion Timing to the new window
#     out_spikes=out_spikes_padded[channel]
#     out_spikes=[out_spike if out_spike >= window[0] and out_spike <= window[-1] else 0 for out_spike in out_spikes]
# else:
#     ripples_start=ripples_start-TOLERANCE


# # Plot the output spikes for the first channel
# plt.figure(figsize=(12, 6))
# plt.scatter(out_spikes, np.zeros_like(out_spikes),marker='o', label=f'Output Spikes (Channel {channel})')
# up_spike_times = np.where(data[:,channel, 0] == 1)[0]
# down_spike_times = np.where(data[:,channel, 1] == 1)[0]
# plt.scatter(up_spike_times, np.ones_like(up_spike_times) * 0.5, marker='^', color='blue', label=f'UP Spikes (Channel {channel})')
# plt.scatter(down_spike_times, np.ones_like(down_spike_times) * -0.5, marker='v', color='red', label=f'DN Spikes (Channel {channel})')
# plt.plot(window,filtered[:, channel], label=f'Filtered Signal (Channel {channel})', color='orange')
# # Plot the GT Insertion Timing
# for ripple_time in ripples_start:
#     plt.axvline(x=ripple_time, color='green', linestyle='--', label='GT Insertion Timing' if ripple_time == ripples_start[0] else "")
# plt.title(f"Output Spikes for Channel {channel} in Dataset {dataset}")
# plt.xlabel("Time (ms)")
# plt.ylabel("Spikes")
# plt.legend()
# plt.grid()
